# Lab 1: Diagnose Overfitting

**Topic 2: Solving Overfitting Issues with Vibe Coding**

In this lab, we will build an intentionally overfitting model on Fashion-MNIST to learn how to identify and diagnose overfitting through training curves.

## Learning Objectives
- Understand what overfitting looks like in training metrics
- Build a model prone to overfitting
- Plot and interpret loss/accuracy curves
- Identify the overfitting onset point

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

## 1. Environment Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

## 2. Load and Preprocess Fashion-MNIST

In [ ]:
# Load Fashion-MNIST dataset
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Class names for Fashion-MNIST
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Normalize pixel values to [0, 1]
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten images for dense layers: (28, 28) -> (784,)
x_train_full_flat = x_train_full.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)

# Split training data into train and validation sets
x_train = x_train_full_flat[:50000]
y_train = y_train_full[:50000]
x_val = x_train_full_flat[50000:]
y_val = y_train_full[50000:]

print(f"Training set: {x_train.shape}")
print(f"Validation set: {x_val.shape}")
print(f"Test set: {x_test_flat.shape}")

## 3. Visualize Sample Data

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_full[i], cmap="gray")
    ax.set_title(class_names[y_train_full[i]])
    ax.axis("off")
plt.suptitle("Sample Fashion-MNIST Images", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Build an Intentionally Overfitting Model

This model has far too many parameters for Fashion-MNIST:
- 5 hidden layers with 512, 512, 256, 256, 128 neurons
- No dropout, no batch normalization, no regularization
- This creates a model with ~600K+ parameters for a relatively simple task

In [ ]:
def build_overfitting_model():
    """Build a dense neural network that is intentionally too large and unregularized."""
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_overfitting_model()
model.summary()

## 5. Train for 30+ Epochs and Capture History

In [ ]:
EPOCHS = 40

history = model.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=1,
)

## 6. Plot Training vs. Validation Curves

In [ ]:
def plot_training_curves(history_dict, title_suffix=""):
    """Plot training vs validation loss and accuracy curves."""
    epochs_range = range(1, len(history_dict["loss"]) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    ax1.plot(epochs_range, history_dict["loss"], "b-", label="Training Loss", linewidth=2)
    ax1.plot(epochs_range, history_dict["val_loss"], "r-", label="Validation Loss", linewidth=2)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title(f"Training vs Validation Loss{title_suffix}")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax2.plot(epochs_range, history_dict["accuracy"], "b-", label="Training Accuracy", linewidth=2)
    ax2.plot(epochs_range, history_dict["val_accuracy"], "r-", label="Validation Accuracy", linewidth=2)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title(f"Training vs Validation Accuracy{title_suffix}")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

fig = plot_training_curves(history.history)
plt.show()

## 7. Identify the Overfitting Onset Point

In [ ]:
val_losses = history.history["val_loss"]
train_losses = history.history["loss"]
train_accs = history.history["accuracy"]
val_accs = history.history["val_accuracy"]

# Best epoch based on validation loss
best_epoch = np.argmin(val_losses) + 1
best_val_loss = val_losses[best_epoch - 1]
best_val_acc = val_accs[best_epoch - 1]

# Overfitting gap at the final epoch
final_train_acc = train_accs[-1]
final_val_acc = val_accs[-1]
overfitting_gap = final_train_acc - final_val_acc

print("=" * 50)
print("OVERFITTING DIAGNOSIS")
print("=" * 50)
print(f"Best epoch (lowest val loss): {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best validation accuracy: {best_val_acc:.4f}")
print(f"")
print(f"Final training accuracy: {final_train_acc:.4f}")
print(f"Final validation accuracy: {final_val_acc:.4f}")
print(f"Overfitting gap: {overfitting_gap:.4f} ({overfitting_gap*100:.1f}%)")
print(f"")
if overfitting_gap > 0.05:
    print("VERDICT: Significant overfitting detected!")
    print(f"The model began overfitting around epoch {best_epoch}.")
    print(f"Training beyond epoch {best_epoch} only improved training metrics,")
    print(f"while validation performance degraded.")
else:
    print("VERDICT: Minimal overfitting detected.")

In [ ]:
# Visualize overfitting onset
fig, ax = plt.subplots(figsize=(10, 6))

epochs_range = range(1, len(val_losses) + 1)
ax.plot(epochs_range, train_losses, "b-", label="Training Loss", linewidth=2)
ax.plot(epochs_range, val_losses, "r-", label="Validation Loss", linewidth=2)
ax.axvline(x=best_epoch, color="green", linestyle="--", linewidth=2, label=f"Best Epoch ({best_epoch})")

# Shade the overfitting region
ax.axvspan(best_epoch, len(val_losses), alpha=0.1, color="red", label="Overfitting Region")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Overfitting Onset Visualization", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(x_test_flat, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"")
print(f"Note: Test accuracy ({test_acc:.4f}) is close to validation accuracy ({final_val_acc:.4f})")
print(f"but far from training accuracy ({final_train_acc:.4f}), confirming overfitting.")